# Time series clustering

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import wfdb
from scipy.signal import butter, filtfilt, find_peaks
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from timeit import default_timer as timer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import GridSearchCV

ts_features_df = pd.read_csv('../../data/processed/time_series_features.csv')
ts_features_df

In [ ]:
# Convert subject_id and hadm_id to object type
ts_features_df['subject_id'] = ts_features_df['subject_id'].astype('object')
ts_features_df['hadm_id'] = ts_features_df['hadm_id'].astype('object')

# Take only numerical columns for correlation analysis
numerical_cols = ts_features_df.select_dtypes(include=[np.number])

print(f"Numerical columns: {numerical_cols.columns.tolist()}")

# Plot the correlation matrix
fig, ax = plt.subplots(figsize=(20, 18), dpi=300)
correlation_matrix = numerical_cols.corr()

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

print(f"Correlation matrix shape: {correlation_matrix}")

sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm',
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            annot_kws={"size": 9}, ax=ax)
plt.title('Correlation Matrix of Time Series Features', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Keep only numerical columns
numerical_cols = ts_features_df.select_dtypes(include=[np.number])

features = [
    'II_hrv_RMSSD',
    'II_qrs_duration_ms',
    'II_r_amplitude',

    'age'
]

reduced_df = numerical_cols[features]

In [ ]:
# Re plot the correlation matrix after removing correlated features
fig, ax = plt.subplots(figsize=(20, 18), dpi=300)
correlation_matrix = reduced_df.corr()
# Create a mask for the upper triangle
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))   
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm',
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            annot_kws={"size": 9}, ax=ax)
plt.title('Correlation Matrix of Reduced Time Series Features', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Standardize the data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(reduced_df)

In [ ]:
# Apply the elbow method to find the optimal number of clusters
inertia = []
K = range(2, 50)
for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(scaled_data)
    inertia.append(kmeans.inertia_)
# Plot the elbow curve
plt.figure(figsize=(20, 16), dpi=200)
plt.plot(K, inertia, 'bo-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.xticks(K)
plt.grid(True)
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import silhouette_score, make_scorer

# Silhouette requires access to the data (X) and the predictions (labels)
def silhouette_scorer(estimator, X):
    clusters = estimator.predict(X)
    # Safety check: Silhouette needs at least 2 clusters to compute
    if len(set(clusters)) < 2:
        return -1.0 
    return silhouette_score(X, clusters)

In [ ]:
param_grid = {
    'n_clusters': range(2, 15),
    'init': ['k-means++', 'random'], 
    'n_init': [10] # Fix number of re-runs for stability
}

kmeans = KMeans(random_state=42)

# n_jobs=-1 uses all processor cores (speed up)
# verbose=1 lets you see progress
grid_search = GridSearchCV(
    kmeans, 
    param_grid, 
    cv=5, 
    scoring=silhouette_scorer, 
    n_jobs=-1,
    verbose=1
)

print("Starting Grid Search...")
grid_search.fit(scaled_data)

print(f"\nBest Silhouette Score: {grid_search.best_score_:.3f}")
print(f"Best Parameters: {grid_search.best_params_}")

# Retrieve the best model
best_kmeans = grid_search.best_estimator_

In [ ]:
# Run Kmeans with the best parameters
final_kmeans = KMeans(
    n_clusters=best_kmeans.n_clusters,
    init=best_kmeans.init,
    n_init=best_kmeans.n_init,
    random_state=42
)
final_kmeans.fit(scaled_data)

In [ ]:
# Add the cluster labels to the original dataframe
ts_features_df['cluster'] = final_kmeans.labels_

In [ ]:
# Define the specific feature pairs to plot
feature_pairs = [
    ('II_r_amplitude', 'II_hrv_SDNN'),
    ('II_qrs_duration_ms', 'II_hrv_SDNN'),
    ('II_r_amplitude', 'II_hrv_RMSSD'),
    ('II_qrs_duration_ms', 'II_hrv_RMSSD'),
    ('II_r_amplitude', 'V5_hrv_SDNN')
]

num_plots = len(feature_pairs)
cols = 3
rows = max(1, (num_plots + cols - 1) // cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5), dpi=300)
axes = np.atleast_2d(axes)

# Create a consistent color palette mapping for all clusters
unique_clusters = sorted(ts_features_df['cluster'].unique())
colors = sns.color_palette('tab20', n_colors=len(unique_clusters))
palette_dict = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}

for ax, (feat_x, feat_y) in zip(axes.flatten(), feature_pairs):
    sns.scatterplot(
        data=ts_features_df, 
        x=feat_y, 
        y=feat_x, 
        hue='cluster', 
        palette=palette_dict,
        ax=ax,
        alpha=0.7,
        legend=False
    )
    ax.set_title(f'{feat_x} vs {feat_y}', fontsize=12)

# Remove unused subplots
for i in range(num_plots, len(axes.flatten())):
    fig.delaxes(axes.flatten()[i])

# Create a single shared legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette_dict[c], 
                      markersize=10, alpha=0.7, label=f'Cluster {c}') 
           for c in unique_clusters]

fig.legend(handles=handles, title='Cluster', loc='upper right', 
           bbox_to_anchor=(0.99, 0.99), fontsize=10, title_fontsize=12)

plt.suptitle('KMeans features scatter plots', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 0.95, 0.97])
plt.show()

In [ ]:
# Plot all the combinations of the original features colored by cluster (ward) in a single plot
import itertools
# Include 'age' only if not already present
features_to_plot = numerical_cols.columns.tolist()
if 'age' not in features_to_plot:
    features_to_plot.append('age')
feature_combinations = list(itertools.combinations(features_to_plot, 2))
num_combinations = len(feature_combinations)
if num_combinations == 0:
    print("Not enough features to create scatter plot combinations (need at least 2 features).")
else:
    cols = 3
    rows = max(1, (num_combinations + cols - 1) // cols)  # Ensure at least 1 row
    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5), dpi=300)
    axes = np.atleast_2d(axes)  # Ensure axes is always 2D for consistent flattening
    
    # Create a consistent color palette mapping for all clusters
    unique_clusters = sorted(ts_features_df['cluster'].unique())
    colors = sns.color_palette('tab20', n_colors=len(unique_clusters))  # Supports up to 20 colors
    palette_dict = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}
    
    for ax, (feat_x, feat_y) in zip(axes.flatten(), feature_combinations):
        sns.scatterplot(
            data=ts_features_df, 
            x=feat_x, 
            y=feat_y, 
            hue='cluster', 
            palette=palette_dict,  # Use the dictionary for consistent colors
            ax=ax,
            alpha=0.7,
            legend=False  # Disable individual legends
        )
        ax.set_title(f'Scatter Plot of {feat_x} vs {feat_y}', fontsize=12)
    
    # Remove any unused subplots
    for i in range(len(feature_combinations), len(axes.flatten())):
        fig.delaxes(axes.flatten()[i])
    
    # Create a single shared legend using the same palette
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette_dict[c], 
                          markersize=10, alpha=0.7, label=f'Cluster {c}') 
               for c in unique_clusters]
    
    fig.legend(handles=handles, title='Cluster', loc='upper right', 
               bbox_to_anchor=(0.99, 0.99), fontsize=10, title_fontsize=12)
    
    plt.tight_layout(rect=[0, 0, 0.95, 1])  # Leave space for the legend
    plt.show()

## Hierarchical Clustering

In [ ]:
from scipy.cluster.hierarchy import linkage, cophenet
from scipy.spatial.distance import pdist
def get_cophenetic_coefficients(X, methods=['single', 'complete', 'average', 'ward']) -> pd.DataFrame:
    """ 
    Calcola i coefficienti cophenetic per i vari metodi di clustering.
    """
    cophenetic_coefficients_df = pd.DataFrame()

    for method in methods:
        Z = linkage(X, method)
        cophenetic_coefficients, _ = cophenet(Z, pdist(X, metric='euclidean'))
        
        cophenetic_coefficients_df[method] = [cophenetic_coefficients]
    
    return cophenetic_coefficients_df

# Usa X che contiene i dati già scalati con MinMaxScaler
get_cophenetic_coefficients(scaled_data)

In [ ]:
from scipy.cluster.hierarchy import dendrogram

methods = ['single', 'complete', 'average', 'ward']

fig, axs = plt.subplots(2, 2, figsize=(14, 14), dpi=150)

for ax, method in zip(axs.flatten(), methods):
    data_dist = pdist(scaled_data, metric='euclidean')
    data_link = linkage(data_dist, method=method, metric='euclidean')
    res = dendrogram(data_link, truncate_mode='lastp', ax=ax, p=20)
    ax.set_title(f'{method} linkage', fontsize=14, fontweight='bold')
    ax.tick_params(labelrotation=90, labelsize=10)
    ax.set_xlabel('Sample index', fontsize=11, fontweight='bold')
    ax.set_ylabel('Distance', fontsize=11, fontweight='bold')

fig.suptitle("Dendrogram for different hierarchical clustering methods", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.cluster.hierarchy import fcluster

# Perform hierarchical clustering with complete linkage
data_dist = pdist(scaled_data, metric='euclidean')
data_link = linkage(data_dist, method='complete')

# Cut the dendogram to 4 clusters
n_clusters = 4
ts_features_df['cluster'] = fcluster(data_link, t=n_clusters, criterion='maxclust')

### Plot with complete linkage

In [ ]:
# Define the specific feature pairs to plot
feature_pairs = [
    ('II_r_amplitude', 'II_hrv_SDNN'),
    ('II_qrs_duration_ms', 'II_hrv_SDNN'),
    ('II_r_amplitude', 'II_hrv_RMSSD'),
    ('II_qrs_duration_ms', 'II_hrv_RMSSD'),
    ('II_r_amplitude', 'V5_hrv_SDNN')
]

num_plots = len(feature_pairs)
cols = 3
rows = max(1, (num_plots + cols - 1) // cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5), dpi=300)
axes = np.atleast_2d(axes)

# Create a consistent color palette mapping for all clusters
unique_clusters = sorted(ts_features_df['cluster'].unique())
colors = sns.color_palette('tab20', n_colors=len(unique_clusters))
palette_dict = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}

for ax, (feat_x, feat_y) in zip(axes.flatten(), feature_pairs):
    sns.scatterplot(
        data=ts_features_df, 
        x=feat_y, 
        y=feat_x, 
        hue='cluster', 
        palette=palette_dict,
        ax=ax,
        alpha=0.7,
        legend=False
    )
    ax.set_title(f'{feat_x} vs {feat_y}', fontsize=12)

# Remove unused subplots
for i in range(num_plots, len(axes.flatten())):
    fig.delaxes(axes.flatten()[i])

# Create a single shared legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette_dict[c], 
                      markersize=10, alpha=0.7, label=f'Cluster {c}') 
           for c in unique_clusters]

fig.legend(handles=handles, title='Cluster', loc='upper right', 
           bbox_to_anchor=(0.99, 0.99), fontsize=10, title_fontsize=12)

plt.suptitle('HC complete features scatter plots', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 0.95, 0.97])
plt.show()

In [ ]:
# Scatter plot all the combinations of the original features colored by cluster in a single plot 
import itertools

# Include 'age' only if not already present
features_to_plot = numerical_cols.columns.tolist()
if 'age' not in features_to_plot:
    features_to_plot.append('age')
feature_combinations = list(itertools.combinations(features_to_plot, 2))
num_combinations = len(feature_combinations)

if num_combinations == 0:
    print("Not enough features to create scatter plot combinations (need at least 2 features).")
else:
    cols = 3
    rows = max(1, (num_combinations + cols - 1) // cols)  # Ensure at least 1 row
    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5), dpi=300)
    axes = np.atleast_2d(axes)  # Ensure axes is always 2D for consistent flattening
    
    # Create a consistent color palette mapping for all clusters
    unique_clusters = sorted(ts_features_df['cluster'].unique())
    colors = sns.color_palette('tab20', n_colors=len(unique_clusters))  # Supports up to 20 colors
    palette_dict = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}
    
    for ax, (feat_x, feat_y) in zip(axes.flatten(), feature_combinations):
        sns.scatterplot(
            data=ts_features_df, 
            x=feat_x, 
            y=feat_y, 
            hue='cluster', 
            palette=palette_dict,  # Use the dictionary for consistent colors
            ax=ax,
            alpha=0.7,
            legend=False  # Disable individual legends
        )
        ax.set_title(f'Scatter Plot of {feat_x} vs {feat_y}', fontsize=12)
    
    # Remove any unused subplots
    for i in range(len(feature_combinations), len(axes.flatten())):
        fig.delaxes(axes.flatten()[i])
    
    # Create a single shared legend using the same palette
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette_dict[c], 
                          markersize=10, alpha=0.7, label=f'Cluster {c}') 
               for c in unique_clusters]
    
    fig.legend(handles=handles, title='Cluster', loc='upper right', 
               bbox_to_anchor=(0.99, 0.99), fontsize=10, title_fontsize=12)
    
    plt.tight_layout(rect=[0, 0, 0.95, 1])  # Leave space for the legend
    plt.show()

## PLot with ward

In [ ]:
from scipy.cluster.hierarchy import fcluster

# Perform hierarchical clustering with ward
data_dist = pdist(scaled_data, metric='euclidean')
data_link = linkage(data_dist, method='ward')

# Cut the dendrogram to get cluster labels (use the same number of clusters as KMeans for comparison)
n_clusters = 3
ts_features_df['cluster'] = fcluster(data_link, t=n_clusters, criterion='maxclust')

In [ ]:
# Define the specific feature pairs to plot
feature_pairs = [
    ('II_r_amplitude', 'II_hrv_SDNN'),
    ('II_qrs_duration_ms', 'II_hrv_SDNN'),
    ('II_r_amplitude', 'II_hrv_RMSSD'),
    ('II_qrs_duration_ms', 'II_hrv_RMSSD'),
    ('II_r_amplitude', 'V5_hrv_SDNN')
]

num_plots = len(feature_pairs)
cols = 3
rows = max(1, (num_plots + cols - 1) // cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5), dpi=300)
axes = np.atleast_2d(axes)

# Create a consistent color palette mapping for all clusters
unique_clusters = sorted(ts_features_df['cluster'].unique())
colors = sns.color_palette('tab20', n_colors=len(unique_clusters))
palette_dict = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}

for ax, (feat_x, feat_y) in zip(axes.flatten(), feature_pairs):
    sns.scatterplot(
        data=ts_features_df, 
        x=feat_y, 
        y=feat_x, 
        hue='cluster', 
        palette=palette_dict,
        ax=ax,
        alpha=0.7,
        legend=False
    )
    ax.set_title(f'{feat_x} vs {feat_y}', fontsize=12)

# Remove unused subplots
for i in range(num_plots, len(axes.flatten())):
    fig.delaxes(axes.flatten()[i])

# Create a single shared legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette_dict[c], 
                      markersize=10, alpha=0.7, label=f'Cluster {c}') 
           for c in unique_clusters]

fig.legend(handles=handles, title='Cluster', loc='upper right', 
           bbox_to_anchor=(0.99, 0.99), fontsize=10, title_fontsize=12)

plt.suptitle('HC ward features scatter plots', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 0.95, 0.97])
plt.show()

In [ ]:

# Include 'age' only if not already present
features_to_plot = numerical_cols.columns.tolist()
if 'age' not in features_to_plot:
    features_to_plot.append('age')
feature_combinations = list(itertools.combinations(features_to_plot, 2))
num_combinations = len(feature_combinations)
if num_combinations == 0:
    print("Not enough features to create scatter plot combinations (need at least 2 features).")
else:
    cols = 3
    rows = max(1, (num_combinations + cols - 1) // cols)  # Ensure at least 1 row
    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5), dpi=300)
    axes = np.atleast_2d(axes)  # Ensure axes is always 2D for consistent flattening
    
    # Create a consistent color palette mapping for all clusters
    unique_clusters = sorted(ts_features_df['cluster'].unique())
    colors = sns.color_palette('tab20', n_colors=len(unique_clusters))  # Supports up to 20 colors
    palette_dict = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}
    
    for ax, (feat_x, feat_y) in zip(axes.flatten(), feature_combinations):
        sns.scatterplot(
            data=ts_features_df, 
            x=feat_x, 
            y=feat_y, 
            hue='cluster', 
            palette=palette_dict,  # Use the dictionary for consistent colors
            ax=ax,
            alpha=0.7,
            legend=False  # Disable individual legends
        )
        ax.set_title(f'Scatter Plot of {feat_x} vs {feat_y}', fontsize=12)
    
    # Remove any unused subplots
    for i in range(len(feature_combinations), len(axes.flatten())):
        fig.delaxes(axes.flatten()[i])
    
    # Create a single shared legend using the same palette
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette_dict[c], 
                          markersize=10, alpha=0.7, label=f'Cluster {c}') 
               for c in unique_clusters]
    
    fig.legend(handles=handles, title='Cluster', loc='upper right', 
               bbox_to_anchor=(0.99, 0.99), fontsize=10, title_fontsize=12)
    
    plt.tight_layout(rect=[0, 0, 0.95, 1])  # Leave space for the legend
    plt.show()